# Pseudo power spectrum of stochastic differential equation

In this notebook, we aim to gain intuition on the stochastic differential equation

$$ \ddot{h} + \gamma(t)\dot{t} + \omega^2(t)h = \xi_f(t) $$

by assuming that the coefficients are slowly varying (very smooth) and that the solution at time $t$ is therefore the local harmonic oscillator of frequency $\omega_0\vert_t = \omega(t)$ etc. in addition to the backreaction between different frequencies. Ignoring the latter part, which cannot be solved analytically as far as I know, we can at least gain intuition on how the local harmonic oscillator part behaves. The fourier transformation diagonalizes the differential operator:

$$ \tilde{h}(-\nu^2 + i\nu\gamma_0\vert_t + \omega_0^2\vert_t) = \tilde{\xi}_f,$$

where $\xi_f$ is allowed to vary in time arbitrarily. Therefore,

$$ \tilde{h}(t,\nu) = \frac{\tilde{\xi}_f}{-\nu^2 + i\nu \gamma_0\vert_t + \omega_0^2\vert_t}$$

## What I learned

Plotting this essentially just traces the $\omega(t)$ field in the TF plane. Comparing against the Wigner function of waveform samples, it looks like the Wigner function also just traces the $\omega(t)$ field, however its width and strength over time varies, probably modulated by $\gamma(t)$ and $\xi_f(t)$.

In [39]:
import numpy as np
from jupytext.combine import map_outputs_to_inputs
%matplotlib tk
from phase_III.strain import *
from phase_II.nifty_re_playground.strain_tools import *
from phase_III.strain import *
from phase_III.useful.helpers import draw_and_plot_field_realizations, get_oscillator_sample
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import nifty.re as jft
key = jax.random.key(42)
import matplotlib.pyplot as plt

In [23]:
def time_frequency_waveform(times, force, gamma, omega):
    """
    :param times:   Times corresponding to gamma(t), omega(t), etc.
    :param force:   The driving force in real space
    :param gamma:   gamma(t)
    :param omega:   omega(t)
    :return:
    """
    dt = times[1] - times[0]
    N = len(times)
    nu = jnp.fft.fftfreq(n=N, d=dt)
    force_h = jnp.fft.fft(force)

    nu_cast = nu[:, None]
    force_cast = force_h[:, None]

    gamma0_cast = gamma[None, :]
    omega0_sq_cast = omega[None, :]**2


    grid = force_cast / (-nu_cast**2 + 1j * nu_cast * gamma0_cast + omega0_sq_cast)
    return grid, times, nu


In [64]:
GW150914 = StrainSignalInference(
    key=key,
    event_name="GW150914",
    detector="H1",
    data_duration_of_hdf5_file="4096sec",
    stationarity_time_scale=32,
    e_fac=1,
    r_fac=1,
    alpha_taper_on_data=.1,
    out_name='osc_dlt_later'
)

signal_domain = GW150914.machinery.t_ss
target_domain = GW150914.machinery.t_ds

oscillator_prior_dct = {
    "frequency": {"offset_mean": 1000, "offset_std": (500, 1e-16), "fluctuations": (1, 1), "loglogavgslope": (-6, 1)},  # log fluctuations...
    "damping": {"offset_mean": 0, "offset_std": (5, 1e-16), "fluctuations": (10, 10), "loglogavgslope": (-6, 1)},
    "force": {"offset_mean": 0, "offset_std": (5, 1e-16), "fluctuations": (10, 10), "loglogavgslope": (0, 0.001), },
    "global_amplitude": (1, 1),
    "init_condition": (0., 1),
}

signal_prior = StochasticOscillatorPrior(oscillator_prior_dct, signal_time_domain=signal_domain, localize_force=None, couple_force_to_frequency=False)
oscillator = HarmonicOscillator(signal_domain_times=signal_domain, signal_prior=signal_prior)

Assumed noise stationarity timescale for Welch-average:  32  seconds.
	Constructing 15 windows over which we average.
	Mean variance of tapered windows: 4.54237106804325
	Compare with area under welch ps:  4.5444512277217575
Hermitian symmetry OK. Max difference between pos and neg is  4.440892098500626e-16

Waveform query for GW150914. Found 1 local matche(s):
	0: 	IGWN-GWTC2p1-v2-GW150914_095045_PEDataRelease_mixed_cosmo.h5
Using  IGWN-GWTC2p1-v2-GW150914_095045_PEDataRelease_mixed_cosmo.h5


2026-04-06  11:47:01 PESummary WARNING : Could not find f_start in input file and one was not passed from the command line. Using 20.0Hz as default


Waveform model bank names:  C01:IMRPhenomXPHM C01:Mixed C01:SEOBNRv4PHM
Using  IMRPhenomXPHM  model.
Input gps center:  1126259462.4  maximum likelihood merger time from template:  1126259462.4241831

You are trying to set up the frequency prior for the oscillator with mean 1000.0±(500, 1e-16). To ensure positivity, 
we are exponentiating internally and therefore changing the mean and standard deviation values to 
 13.82±((Array(0.81, dtype=float64), Array(0., dtype=float64))).
If you are unsure this has the desired effect, check samples via `StochasticOscillatorPrior.plot_omega_samples`.


In [17]:
# oscillator.plot_samples(1, key, show_spectrogram=True)

In [18]:
t = oscillator.evolution_times

In [19]:
key, waveform_sample, omega_sample, gamma_sample, force_sample = get_oscillator_sample(oscillator_model=oscillator, key=key)

In [20]:
omega_sample

Array([4464.77172588, 4466.66949439, 4475.2156367 , ...,
       4473.21434213, 4472.8955001 , 4463.12172861],      dtype=float64)

In [31]:
_ = plt.figure()
plt.plot(t, omega_sample)
plt.show()

In [47]:
grid, times, freqs = time_frequency_waveform(times=oscillator.evolution_times, force=force_sample, gamma=gamma_sample, omega=omega_sample)

In [48]:
np_grid = np.array(grid)
np_grid[jnp.abs(grid) > 1e-3 ] = 0

In [52]:
def custom_vis_stress(stress_matrix, rows, cols, smooth=False, smoothing_level=5, custom_ax=None, delay_plot=False,
                     plot_colorbar=True, colorbar_label="Stress", cmap="plasma", tl="", hlines=None, vlines=None,
                     xl=r"Time $\mathrm{[s]}$", yl=r"Frequency $\mathrm{[Hz]}$", return_aux=False, **kwargs):
    if custom_ax is None:
        plt.figure(figsize=(4., 4.))
        ax = plt.gca()
    else:
        ax = custom_ax
    stress_matrix = stress_matrix.real

    cols_are_increasing = np.all(np.diff(cols) > 0)  # strictly increasing
    rows_are_increasing = np.all(np.diff(rows) > 0)  # strictly increasing
    if not cols_are_increasing:
        raise ValueError("Columns must be increasing")
    if not rows_are_increasing:
        stress_matrix = np.fft.fftshift(stress_matrix, axes=0)  # shift DC frequency to middle
        rows = np.fft.fftshift(rows, axes=0)
        print("\t\tRows must be increasing, assuming a priori standard DFT order and moving DC to the middle")
        # Must be increasing because we want to plot from - frequency to 0 to + frequency on the y-axis

    if smooth:
        stress_matrix = smooth_matrix(stress_matrix, smoothing_level)

    im = ax.imshow(stress_matrix, origin='lower', aspect='auto',
               extent=[np.min(cols), np.max(cols), np.min(rows), np.max(rows)],
               cmap=cmap, interpolation='nearest',   # nearest: No smoothing
               norm='log')

    if hlines is not None:
        ax.hlines(hlines, 0, np.max(cols), color="r", ls="-")
    if vlines is not None:
        ax.vlines(vlines, 0, np.max(rows), color="r", ls="-")

    if plot_colorbar:
        cb = ax.colorbar(label=colorbar_label)
    else:
        cb = None

    if not delay_plot:
        thesis_plot(mode="square", xl=xl, yl=yl, title=tl, custom_ax=ax, tight_ly=False, **kwargs)

    if return_aux:
        aux = (cb, im)
        return aux

In [53]:
custom_vis_stress(stress_matrix=jnp.abs(grid), rows=freqs, cols=times, smooth=False)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


In [75]:
import matplotlib.pyplot as plt
import jax.numpy as jnp

def multi_plot(times, freqs,
               omega_sample,
               gamma_sample,
               force_sample,
               waveform_sample,
               grid,
               show_tf=True):

    fig, axes = plt.subplots(5, 2, figsize=(12, 14))

    Xs = [
        omega_sample,
        gamma_sample,
        force_sample,
        jnp.fft.fft(force_sample),
        waveform_sample
    ]

    titles = [
        "omega(t)",
        "gamma(t)",
        "force(t)",
        "FFT(force)",
        "waveform"
    ]

    # left column
    for i, (X, title) in enumerate(zip(Xs, titles)):
        axes[i, 0].plot(times, jnp.real(X))
        axes[i, 0].set_title(title)

    if show_tf:
        # 1) grid-based TF (row 0, right)
        custom_vis_stress(
            stress_matrix=jnp.abs(grid),
            rows=freqs,
            cols=times,
            smooth=False,
            custom_ax=axes[0, 1],
            delay_plot=True,
        )

        # 2) waveform TF (row 1, right)
        S, t_loc, f_loc = Stress_jft(waveform_sample, time=times)

        visualize_stress(
            stress_matrix=S,
            rows=f_loc,
            cols=t_loc,
            smooth=True,
            custom_ax=axes[1, 1],
            delay_plot=True,
        )

        # 3) |waveform TF| in logscale (row 1, right)
        custom_vis_stress(
            stress_matrix=jnp.abs(S),
            rows=f_loc,
            cols=t_loc,
            smooth=True,
            custom_ax=axes[2, 1],
            delay_plot=True,
        )

        # hide rows 3–4 on right
        for i in range(3, 5):
            axes[i, 1].axis("off")

    else:
        for i in range(5):
            axes[i, 1].axis("off")

    plt.show()

In [77]:
for _ in range(10):

    key, waveform_sample, omega_sample, gamma_sample, force_sample = get_oscillator_sample(oscillator_model=oscillator, key=key)

    grid, times, freqs = time_frequency_waveform(times=oscillator.evolution_times, force=force_sample, gamma=gamma_sample, omega=omega_sample)

    multi_plot(times=times, freqs=freqs, omega_sample=omega_sample, gamma_sample=gamma_sample, force_sample=force_sample, grid=grid, waveform_sample=waveform_sample, show_tf=True)

    plt.show(block=True)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


KeyboardInterrupt: 